# 02 — Disease Classification with EfficientNet-B0

This notebook demonstrates the full training pipeline and detailed evaluation of the transfer-learning classifier on the PlantVillage dataset.

**Approach**:
- Architecture: **EfficientNet-B0** pre-trained on ImageNet (via `timm`)
- Phase 1: Freeze backbone → train only the classifier head (fast convergence)
- Phase 2: Unfreeze all layers → fine-tune with 10× smaller learning rate
- Mixed-precision (AMP) when CUDA is available
- Evaluation: accuracy, per-class F1, confusion matrix, **Grad-CAM**

Expected accuracy: **>95%** (literature with GoogLeNet: 99.35%)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

from src.dataset import load_plantvillage
from src.model   import build_model, freeze_backbone, unfreeze_all, get_gradcam_target_layer
from src.utils   import (
    denormalize, plot_training_curves, compute_gradcam,
    apply_gradcam, plot_gradcam_grid
)

plt.rcParams['figure.dpi'] = 110
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}  |  AMP: {USE_AMP}')

## 1. Load dataset

In [ ]:
train_ds, test_ds, class_names = load_plantvillage(config='color')

BATCH_SIZE   = 32
NUM_WORKERS  = 0    # Set to 0 on Windows; increase on Linux/macOS

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')
print(f'Classes: {len(class_names)}')

## 2. Build model

In [ ]:
MODEL_NAME = 'efficientnet_b0'
model      = build_model(num_classes=len(class_names), model_name=MODEL_NAME).to(DEVICE)

total_p    = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_p:,}')

# Preview the model's final classifier layer
print('\nClassifier head:')
print(model.classifier)

## 3. Training

> **Note**: Training on CPU takes several hours.  
> On a GPU (e.g., Google Colab T4) it takes ~25–35 min for 20 epochs.  
> If you already ran `train.py`, skip to **Section 4** and load the checkpoint.

In [ ]:
PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 10
LR            = 1e-3
criterion     = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler        = torch.cuda.amp.GradScaler(enabled=USE_AMP)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0


def run_epoch(model, loader, criterion, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad if train else torch.no_grad
    with ctx():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out  = model(imgs)
                loss = criterion(out, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            preds = out.argmax(1)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, all_preds, all_labels


print('=== Phase 1: head warm-up (backbone frozen) ===')
freeze_backbone(model)
opt1 = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
sch1 = CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS)

for epoch in range(1, PHASE1_EPOCHS + 1):
    tl, ta, _, _       = run_epoch(model, train_loader, criterion, opt1,  train=True)
    vl, va, vp, vlbls  = run_epoch(model, test_loader,  criterion, train=False)
    sch1.step()
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    print(f'[P1 {epoch}/{PHASE1_EPOCHS}] loss {tl:.4f}/{vl:.4f}  acc {ta:.4f}/{va:.4f}')
    if va > best_val_acc:
        best_val_acc = va
        best_preds, best_labels = vp, vlbls
        os.makedirs('../runs/notebook', exist_ok=True)
        torch.save(model.state_dict(), '../runs/notebook/best_model.pth')

print()
print('=== Phase 2: full fine-tuning ===')
unfreeze_all(model)
opt2 = AdamW(model.parameters(), lr=LR / 10)
sch2 = CosineAnnealingLR(opt2, T_max=PHASE2_EPOCHS)

for epoch in range(1, PHASE2_EPOCHS + 1):
    tl, ta, _, _       = run_epoch(model, train_loader, criterion, opt2,  train=True)
    vl, va, vp, vlbls  = run_epoch(model, test_loader,  criterion, train=False)
    sch2.step()
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    print(f'[P2 {epoch}/{PHASE2_EPOCHS}] loss {tl:.4f}/{vl:.4f}  acc {ta:.4f}/{va:.4f}')
    if va > best_val_acc:
        best_val_acc = va
        best_preds, best_labels = vp, vlbls
        torch.save(model.state_dict(), '../runs/notebook/best_model.pth')
        print(f'  ✓ Best: {best_val_acc:.4f}')

with open('../runs/notebook/history.json', 'w') as f:
    json.dump(history, f)
with open('../runs/notebook/class_names.json', 'w') as f:
    json.dump(class_names, f)

print(f'\nBest validation accuracy: {best_val_acc:.4f}')

## 4. Load checkpoint (if training was skipped)

In [ ]:
# Run this cell only if you skipped training (e.g., ran train.py separately)
RUN_DIR = '../runs/exp1'   # adjust to your run directory

if not 'best_preds' in dir():
    with open(os.path.join(RUN_DIR, 'class_names.json')) as f:
        class_names = json.load(f)
    with open(os.path.join(RUN_DIR, 'history.json')) as f:
        history = json.load(f)

    model = build_model(num_classes=len(class_names), model_name=MODEL_NAME).to(DEVICE)
    model.load_state_dict(torch.load(os.path.join(RUN_DIR, 'best_model.pth'), map_location=DEVICE))
    model.eval()

    criterion = nn.CrossEntropyLoss()
    _, _, best_preds, best_labels = run_epoch(model, test_loader, criterion, train=False)

    print('Checkpoint loaded.')
else:
    print('Using model trained in this session.')

## 5. Training curves

In [ ]:
fig = plot_training_curves(history)
plt.savefig('training_curves.png', bbox_inches='tight')
plt.show()

## 6. Classification report

In [ ]:
print(classification_report(best_labels, best_preds, target_names=class_names, digits=3))

## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(best_labels, best_preds)

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    cm, annot=False, fmt='d',
    xticklabels=class_names, yticklabels=class_names,
    cmap='Blues', ax=ax, linewidths=0.3
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title('Confusion Matrix — PlantVillage Test Set', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()

## 8. Top misclassifications

In [ ]:
from collections import Counter

errors = [
    (class_names[t], class_names[p])
    for t, p in zip(best_labels, best_preds)
    if t != p
]
top_errors = Counter(errors).most_common(10)

print(f'Total test errors: {len(errors)} / {len(best_preds)} '
      f'({100*len(errors)/len(best_preds):.1f}%)')
print('\nTop-10 confusion pairs (True → Predicted):')
for (true_c, pred_c), cnt in top_errors:
    print(f'  {cnt:3d}×  {true_c:<40} → {pred_c}')

## 9. Grad-CAM visualisation

Grad-CAM shows **which regions** of the leaf the model uses to make its prediction.  
Hot colours (red/yellow) indicate high importance.

In [ ]:
import torch.nn.functional as F

model.eval()
target_layer = get_gradcam_target_layer(model, MODEL_NAME)
print(f'Grad-CAM target layer: {type(target_layer).__name__}')

# Collect 8 test samples — spread across different classes
N_VIS    = 8
batch_tensors, batch_labels = next(iter(test_loader))
selected = batch_tensors[:N_VIS]
sel_lbls = batch_labels[:N_VIS]

tensors_vis    = []
heatmaps_vis   = []
predictions_vis = []
gt_vis         = []

for i in range(N_VIS):
    tensor = selected[i]
    heatmap = compute_gradcam(
        model, tensor, target_layer,
        class_idx=None,
        device=str(DEVICE)
    )

    with torch.no_grad():
        logits = model(tensor.unsqueeze(0).to(DEVICE))
        pred_idx = logits.argmax(1).item()

    tensors_vis.append(tensor)
    heatmaps_vis.append(heatmap)
    predictions_vis.append(class_names[pred_idx].split('___')[1] if '___' in class_names[pred_idx] else class_names[pred_idx])
    gt_vis.append(class_names[sel_lbls[i].item()].split('___')[1] if '___' in class_names[sel_lbls[i].item()] else class_names[sel_lbls[i].item()])

fig = plot_gradcam_grid(tensors_vis, heatmaps_vis, predictions_vis, gt_vis, figsize=(22, 6))
plt.savefig('gradcam_grid.png', bbox_inches='tight')
plt.show()

In [ ]:
# Single detailed Grad-CAM example
idx    = 0
tensor = selected[idx]
orig   = denormalize(tensor)

heatmap = compute_gradcam(model, tensor, target_layer, device=str(DEVICE))
overlay = apply_gradcam(orig, heatmap)

with torch.no_grad():
    logits = model(tensor.unsqueeze(0).to(DEVICE))
    probs  = F.softmax(logits, dim=1)[0]
pred_class = class_names[probs.argmax().item()]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(orig);     axes[0].set_title('Original');     axes[0].axis('off')
axes[1].imshow(overlay);  axes[1].set_title(f'Grad-CAM\nPred: {pred_class}'); axes[1].axis('off')
im = axes[2].imshow(heatmap, cmap='jet'); axes[2].set_title('Heatmap (raw)'); axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.suptitle(f'Ground truth: {class_names[sel_lbls[idx].item()]}', fontsize=11)
plt.tight_layout()
plt.savefig('gradcam_detail.png', bbox_inches='tight')
plt.show()

---
## Summary

| Metric | Value |
|--------|-------|
| Architecture | EfficientNet-B0 |
| Parameters | ~5.3M |
| Training images | 43,596 |
| Test images | 10,709 |
| Expected val accuracy | >95% |
| Interpretability | Grad-CAM |

**Next**: `03_leaf_area.ipynb` — leaf segmentation and disease area quantification.